# 04 - QAOA Portfolio Optimization: Classical vs Quantum

**Module 2 - Financial Portfolio Optimization Platform**

Selecting `budget` assets out of `num_assets` to minimise a risk-adjusted cost is a combinatorial optimization problem. We encode it as a **QUBO** (Quadratic Unconstrained Binary Optimization), convert to an Ising Hamiltonian, and solve it two ways:

- **Classical**: brute force - exact and guaranteed optimal (feasible for small N).
- **Quantum**: **QAOA** (Quantum Approximate Optimization Algorithm) - a variational circuit whose parameters are tuned to drive measurements toward low-cost bitstrings.

The platform uses **simulated financial data** (per the specification) via `generate_simulated_market_data()`, then runs `run_portfolio_optimization()` from `backend/app/optimization/qaoa_portfolio.py`.

**Note:** on a simulator QAOA does not beat brute force - the value is demonstrating that the quantum method *matches* the exact optimum and would scale to problem sizes where brute force becomes intractable on future hardware.

In [1]:
# Make the platform's backend package importable from this notebook.
# The notebooks call the SAME code the FastAPI app runs, so results here
# and results in the deployed platform are guaranteed identical.
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join('..', 'backend')))
print('Backend path wired. Python', sys.version.split()[0])

Backend path wired. Python 3.14.4


In [2]:
from app.optimization.qaoa_portfolio import run_portfolio_optimization

result = run_portfolio_optimization(
    num_assets=4,
    budget=2,            # select exactly 2 of the 4 assets
    risk_aversion=0.5,
    seed=42,
    trials=3,
)
print('Market data (simulated):')
for name, r, v in zip(result['market_data']['asset_names'],
                      result['market_data']['expected_returns'],
                      result['market_data']['volatilities']):
    print(f'  {name}: expected_return={r:+.4f}  volatility={v:.4f}')

Market data (simulated):
  ASSET_A: expected_return=+0.1484  volatility=0.1235
  ASSET_B: expected_return=+0.1014  volatility=0.3439
  ASSET_C: expected_return=+0.1602  volatility=0.2903
  ASSET_D: expected_return=+0.1376  volatility=0.2965


### Classical (brute-force) solution - exact optimum

In [3]:
cs = result['classical_solution']
print('Method      :', cs['method'])
print('Bitstring   :', cs['bitstring'])
print('QUBO cost   :', cs['qubo_cost'])
print('Assets sel. :', cs['assets_selected'], '(constraint satisfied:', cs['cardinality_constraint_satisfied'], ')')

Method      : Brute force (exact, guaranteed optimal)
Bitstring   : 0101
QUBO cost   : -8.253
Assets sel. : 2 (constraint satisfied: True )


### Quantum (QAOA) solution and comparison

In [4]:
qs = result['quantum_solution']
comp = result['comparison']
print('QAOA bitstring       :', qs['bitstring'])
print('QAOA QUBO cost       :', qs['qubo_cost'])
print()
print('Quantum matched classical optimum :', comp['quantum_matched_classical_optimum'])
print('Cost difference                   :', comp['cost_difference'])

QAOA bitstring       : 0101
QAOA QUBO cost       : -8.253

Quantum matched classical optimum : True
Cost difference                   : -0.0


### Repeated-evaluation statistics (QAOA is stochastic)

QAOA is a probabilistic method, so we run several independent trials on identical inputs and report how often the exact optimum was recovered.

In [5]:
rep = result['repeated_evaluation']
print('Trials run                     :', rep['trials_run'])
print('Times matched exact optimum    :', rep['times_matched_classical_optimum'])
print('Optimum recovery rate          :', rep['optimum_recovery_rate'])
print('Cardinality satisfaction rate  :', rep['cardinality_satisfaction_rate'])
print('QUBO cost best / mean / worst  :', rep['qubo_cost_best'], '/', rep['qubo_cost_mean'], '/', rep['qubo_cost_worst'])
print()
print('Per-trial records:')
for t in rep['per_trial']:
    print(' ', t)

Trials run                     : 3
Times matched exact optimum    : 3
Optimum recovery rate          : 1.0
Cardinality satisfaction rate  : 1.0
QUBO cost best / mean / worst  : -8.253 / -8.253 / -8.253

Per-trial records:
  {'trial': 1, 'bitstring': '0101', 'qubo_cost': -8.253, 'assets_selected': 2, 'cardinality_constraint_satisfied': True, 'matched_classical_optimum': True, 'iterations_run': 26}
  {'trial': 2, 'bitstring': '0101', 'qubo_cost': -8.253, 'assets_selected': 2, 'cardinality_constraint_satisfied': True, 'matched_classical_optimum': True, 'iterations_run': 30}
  {'trial': 3, 'bitstring': '0101', 'qubo_cost': -8.253, 'assets_selected': 2, 'cardinality_constraint_satisfied': True, 'matched_classical_optimum': True, 'iterations_run': 30}
